In [1]:
from typing import Any, Dict, List, Optional

from pydantic import BaseModel

from agente_avaliacao_imagens.schemas import AnaliseImagens, FeedbackImagens


class ReActInput(BaseModel):
    """Entrada do agente ReAct de análise de imagens."""

    fotos_urls: List[str] = []
    api_key: Optional[str] = None

c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from phoenix.otel import register

tracer_provider = register(
  project_name="agente-react-imoveis",
  auto_instrument=True
)

08/05/2026 01:31:42 PM 📋 Ensuring phoenix working directory: C:\Users\jefer\.phoenix
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.schemas
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.tables
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.types
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.constraints
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.defaults
08/05/2026 01:31:48 PM setup plugin alembic.autogenerate.comments


OpenTelemetry Tracing Details
|  Phoenix Project: agente-react-imoveis
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\.venv\Lib\site-packages\phoenix\otel\otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


In [3]:
import logging
from typing import List

from langchain_core.tools import tool

from agente_avaliacao_imagens.prompts import PROMPT_DESCREVER_FOTO
from agente_avaliacao_imagens.utils import processar_todos_lotes

logger = logging.getLogger(__name__)


@tool
async def descrever_fotos(fotos_urls: List[str]) -> str:
    """Processa as fotos do imóvel e devolve a descrição técnica de cada imagem.

    Use esta ferramenta para obter a descrição das fotos. Depois, com base nela,
    preencha a análise estruturada final (scores, problemas, pontos fortes).

    Args:
        fotos_urls: lista de URLs das fotos do imóvel.
    """
    if not fotos_urls:
        return "Nenhuma URL de foto fornecida."
    
    logger.info(f"Processando {len(fotos_urls)} fotos para descrição.")

    descricao = await processar_todos_lotes(fotos_urls, 5, prompt=PROMPT_DESCREVER_FOTO)
    if not descricao:
        logger.error("Nao foi possivel descrever as fotos.")
        return "Falha ao descrever as fotos."
    return descricao

In [14]:
import json
import logging
import os
from typing import List, Optional

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.prebuilt import create_react_agent

from agente_avaliacao_imagens.schemas import AnaliseImagens
#from .tools import descrever_fotos

logger = logging.getLogger(__name__)

MODELO_AGENTE = os.getenv("MODELO_AGENTE_IMAGENS",
                          #"z-ai/glm-5.2"#V
                          "meta/llama-3.1-8b-instruct"V
                          #"nvidia/nemotron-3-ultra-550b-a55b" #V
                          # "google/gemma-4-31b-it" X
                          #"poolside/laguna-xs-2.1" #X
                          #"qwen/qwen-2.5-72b-instruct" #X
                          ) #"moonshotai/kimi-k2.6")"deepseek-ai/deepseek-v4-flash")

SYSTEM_PROMPT = """Você é um engenheiro civil e especialista em avaliação de imóveis para house flipping.

Sua tarefa é analisar as fotos de um imóvel e produzir um relatório técnico estruturado.

Passos:
1. Chame a ferramenta `descrever_fotos` com as URLs das fotos. Ela devolverá a descrição técnica de cada imagem.
2. Analise a descrição recebida e preencha a análise estruturada final com os campos abaixo.

Definição de cada campo do relatório final:

- **score_conservacao (float 0-10):** condição geral de conservação/mainutenção do que está visível (infiltrações, trincas, desgaste, estado de paredes/teto).
- **score_acabamento (float 0-10):** qualidade/padrão dos materiais (piso, revestimentos, metais, portas, esquadrias).
- **score_potencial_reforma (float 0-10):** o quanto é viável/vantajoso reformar o espaço (nota alta = boa estrutura que valoriza com melhorias; nota baixa = exige demolição pesada ou já está em ótimo estado).
- **confianca_imagem (float 0-10):** o quanto a descrição é confiável, clara e útil para uma avaliação técnica.
- **imagem_aceitavel (bool):** `true` se a foto mostra elementos reais do imóvel e é clara; `false` se for irrelevante (selfie, parede escura, objeto aleatório) ou a descrição for vaga demais.
- **problemas_visiveis (List[str]):** patologias, defeitos, danos ou sinais de desgaste identificados. Vazio se não houver.
- **pontos_fortes (List[str]):** aspectos positivos observados (iluminação natural, piso em bom estado, acabamento moderno, área espaçosa). Vazio se não houver.
- **observacoes (str):** resumo da opinião técnica. Se `imagem_aceitavel = false` ou as notas forem baixas, use este campo para justificar tecnicamente.

Regras importantes:
- Baseie-se APENAS na descrição fornecida pela ferramenta. Nunca invente ou infira o que não está visível.
- Se um aspecto não puder ser avaliado, pondere as notas de forma neutra e registre a limitação em `observacoes`.
- Se as fotos não retratarem um ambiente de imóvel (imagem irrelevante/ilegível), marque `imagem_aceitavel = false`, atribua `0.0` a todos os scores e explique em `observacoes`.
- Responda SEMPRE em português.
"""


def criar_agente_imagens(api_key: Optional[str] = None):
    model = ChatNVIDIA(
        model=MODELO_AGENTE,
        api_key=api_key or os.getenv("NVIDIA_API_KEY"),
    )
    return create_react_agent(
        model=model,
        tools=[descrever_fotos],
        prompt=SYSTEM_PROMPT,
        response_format=AnaliseImagens,
    )


async def analisar_imagens(
    fotos_urls: List[str],
    api_key: Optional[str] = None,
) -> AnaliseImagens:
    """Executa o agente ReAct de análise de imagens e devolve o relatório estruturado."""
    agente = criar_agente_imagens(api_key=api_key)

    mensagem_usuario = {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Analise as fotos do imóvel:\n"
                    + json.dumps(fotos_urls, ensure_ascii=False, indent=2)
                ),
            }
        ]
    }

    resultado = await agente.ainvoke(mensagem_usuario)
    resposta = resultado.get("structured_response")

    if isinstance(resposta, dict):
        return AnaliseImagens(**resposta)
    return resposta

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1975982450.py, line 16)

In [1]:
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

In [ ]:
fotos = df['fotos'].iloc[6].tolist()

In [ ]:
fotos

['https://resizedimgs.zapimoveis.com.br/img/vr-listing/2be5cbec55da625c7063443f287d852b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/80f6aae4cdef3bb2baf1c8609934408b/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/532e18b9c870fd26cfc2a712304896ab/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/ed7f7a3d250cc07765b1d97fa3e94448/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zapimoveis.com.br/img/vr-listing/fa37d92847ba04c856f07049f497b1c3/apartamento-com-3-quartos-a-venda-105m-no-saguacu-joinville.webp?action=fit-in&dimension=870x707&seo=false',
 'https://resizedimgs.zap

In [ ]:
response

AnaliseImagens(score_conservacao=8.0, score_acabamento=8.0, score_potencial_reforma=5.0, confianca_imagem=8.5, imagem_aceitavel=True, problemas_visiveis=[], pontos_fortes=[' Fachada de edifício', 'Bom estado de conservação', 'Design contemporâneo'], observacoes='A imagem é redundante, repetindo a fachada exibida nas fotos 1 e 5[20, 39]. Analisa apenas o exterior do edifício, que não apresenta danos visíveis[37]. Ausência de informações sobre os ambientes internos do imóvel[41].')

In [3]:
import pandas as pd
import json

In [4]:
#df = pd.read_json('olx_alugueis.json', lines=True)
import pandas as pd
from pathlib import Path


cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent
PASTA_DADOS = BASE_DIR / 'dados' / cidade
df = pd.read_parquet(PASTA_DADOS / f'{cidade}_imoveis_limpo_2026-08.parquet')

with open(PASTA_DADOS/ 'joinville_aluguel_olx_2026-08.json', 'r', encoding='utf-8') as f:
    df = json.load(f)

In [7]:
df = pd.read_parquet("chaves_mao_alugueis.parquet")

In [8]:
df

,url,titulo,metragem,banheiros,vagas,quartos,valor_imovel,condominio,endereco,iptu,descricao,data_criacao,caracteristicas,fotos,link_maps
0,https://www.chavesnamao.com.br/imovel/casa-a-v...,"Casa com 3 quartos à venda na Rua Otto Benack,...",325m²,2,2,3,1400000,0,"Rua Otto Benack, 84, Bom Retiro, Joinville/SC",152,"Lindo Sobrado à venda no bairro Bom Retiro, ed...",None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
1,https://www.chavesnamao.com.br/imovel/casa-a-v...,"Casa para Venda em Joinville, Boehmerwald, 3 d...",502m²,2,2,3,319000,0,"Rua Faustino Busarello, 847, Boehmerwald, Join...",0,Terreno com 2 Casas e Área de Festas Excelente...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
2,https://www.chavesnamao.com.br/imovel/apartame...,"Apartamento à venda por R$ 1.621.080,69 - Sant...",None,4,2,3,1621080,1800,"Endereço Indisponível \nSanto Antônio, Joinvi...",0,Este empreendimento de alto padrão no bairro S...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
3,https://www.chavesnamao.com.br/imovel/terreno-...,Terreno em condomínio fechado à venda na Rua S...,180m²,0,None,None,298000,0,"Rua Severino Gretter, 306, Espinheiros, Joinvi...",0,* TERRENO EM CONDOMINIO FECHADO A VENDA NO BAI...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
4,https://www.chavesnamao.com.br/imovel/terreno-...,"Terreno à venda na Vila Nova, Joinville",718m²,0,None,None,980000,0,"Endereço Indisponível \nVila Nova, Joinville/ SC",0,"Terreno Comercial a Venda – Vila Nova | 718,20...",None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
5,https://www.chavesnamao.com.br/imovel/casa-a-v...,"Casa / Sobrado para Venda em Joinville, Profip...",243m²,5,4,5,850000,0,"Rua Urânio, 79, Profipo, Joinville/SC",0,Belo Sul Imóveis vende casa de dois pavimentos...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
6,https://www.chavesnamao.com.br/imovel/terreno-...,"Terreno à venda, 390 m² por R$ 319.000,00 - Bo...",390m²,0,None,None,319000,0,"Endereço Indisponível \nBom Retiro, Joinville...",0,Terreno à Venda no Bairro Bom Retiro Joinville...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
7,https://www.chavesnamao.com.br/imovel/apartame...,Apartamento com 2 quartos à venda na Rua Babit...,62m²,2,2,2,550000,0,"Rua Babitonga, Floresta, Joinville/SC",0,"Se você busca conforto, segurança e um imóvel ...",None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
8,https://www.chavesnamao.com.br/imovel/casa-a-v...,Oportunidade! Geminado à Venda no Espinheiros ...,63m²,1,1,2,395000,0,"Endereço Indisponível \nEspinheiros, Joinvill...",500,Excelente opção para quem busca um imóvel comp...,None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
9,https://www.chavesnamao.com.br/imovel/apartame...,Residencial Solar de Madrid - Apartamento ampl...,111m²,2,1,3,750000,737,"Endereço Indisponível \nCentro, Joinville/ SC",400,"Localizado no coração da cidade, o Residencial...",None,[],[https://www.chavesnamao.com.br/imn/1200x0800/...,https://www.google.com/maps/embed/v1/place?key...
